<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/iLogos/logo_novafct.png" width="200">

# Departamento de Engenharia Mecânica e Industrial
## Mecânica dos Sólidos II

## Tensões tangenciais em vigas com carregamentos transversais

### Problema 3

A figura representa a secção transversal de uma viga de madeira à qual foram aparafusadas duas placas de aço com 10 mm de espessura, utilizando para tal parafusos com 12 mm de diâmetro e espaçamento longitudinal de 200 mm. Sabe-se que os módulos de elasticidade da madeira e do aço são, respetivamente, $E$ = 10 GPa e $E$ = 210 GPa, que as tensões tangenciais admissíveis na placa de aço e na madeira são, respetivamente, $\tau_\mathrm{adm, aço}$ = 135 MPa e $\tau_\mathrm{adm, madeira}$ = 7 MPa e ainda que a força de corte máxima nos parafusos é $F_\mathrm{max}$ = 15 kN. Determine o esforço transverso máximo que a secção transversal suporta.

<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/Notebooks/Au05/P3/MSII_Au05_P3.png"
style="max-height: 100%; max-width: 100%;"/>

### Resolução

In [26]:
import numpy as np
import sympy as sy
from sympy.solvers import solve
import matplotlib.pyplot as plt
import os

cor = '2'
if cor == '1':
    plt.rcParams['axes.facecolor'] = (.15, .15, .15)
    plt.rcParams['figure.facecolor'] = (.15, .15, .15)
    plt.rcParams['font.family'] = 'monospace'
    plt.rcParams['font.size'] = 18
    # plt.rcParams['text.usetex'] = True
    params = {"ytick.color" : (.8, .8, .8),
              "xtick.color" : (.8, .8, .8),
              "grid.color" : (.2, .2, .2),
              "text.color" : (.7, .7, .7),
              "axes.labelcolor" : (.8, .8, .8),
              "axes.edgecolor" : (.15, .15, .15)}
else:
    plt.rcParams['axes.facecolor'] = (.7, .7, .7)
    plt.rcParams['figure.facecolor'] = (.7, .7, .7)
    plt.rcParams['font.family'] = 'monospace'
    plt.rcParams['font.size'] = 18
    # plt.rcParams['text.usetex'] = True
    params = {"ytick.color" : (.1, .1, .1),
              "xtick.color" : (.1, .1, .1),
              "grid.color" : (.2, .2, .2),
              "text.color" : (.1, .1, .1),
              "axes.labelcolor" : (.1, .1, .1),
              "axes.edgecolor" : (.15, .15, .15)}
plt.rcParams.update(params)

# data structure, units: N, mm, MPa
# Create an empty class
class varin: pass

w = varin()
s = varin()
d = varin()

w.b = 150.
w.h = 250.
w.E = 10.e3 # unit: MPa
w.tauadm = 7 # unit: MPa

s.b = 150.
s.h = 10.
s.E = 210.e3 # unit: MPa
s.tauadm = 135 # unit: MPa

d.ht = w.h + 2*s.h
d.padia = 12. # mm
d.palen = 200. # mm
d.Fcomax = 15.e3 # N

#### Propriedades da secção transversal homogénea equivalente

- Razão de módulos considerando a secção transformada em madeira:

\begin{equation*}
n = \frac{E_\mathrm{streel}}{E_\mathrm{wood}}
\end{equation*}

In [27]:
n = s.E/w.E
print(f'n = Es/Ew = {n:.1f}')

n = Es/Ew = 21.0


- Mantendo a altura da viga (viga composta em altura), qual será a área da região transformada de aço em madeira?

In [28]:
b_s2w = s.b*n
print(f'b_s2w = {b_s2w:.1f} [mm]')
A_s2w = b_s2w*s.h
print(f'A_s2w = {A_s2w:.1f} [mm²]')

b_s2w = 3150.0 [mm]
A_s2w = 31500.0 [mm²]


- Para a secção homogénea equivalente feita apenas de madeira, onde está o centroóide? E qual é o momento de inércia da secção em relação ao eixo $z$?

Pela geometria da secção, o centróide é facilmente determinado no centro geométrico do perfil.
O momento de inércia pode ser obtido por operações booleanas e recorrendo, eventualmente, ao teorema dos eixos paralelos.

In [29]:
def irec(i,j): return i*j**3/12

print('alma:')
b1, h1 = w.b, w.h
Iz1 = irec(b1, h1)
print(f'Iz (alma) :: (b,h) = ({b1:.1f},{h1:.1f}) :: {Iz1:.3e} [mm⁴]')

print('banzo:')
b2, h2 = b_s2w, s.h
Izc2 = irec(b2, h2)
print(f'Iz (banzo) :: (b,h) = ({b2:.0f},{h2:.0f}) ::  {Izc2:.3e} [mm⁴]')
A2 = b_s2w*s.h
print(f'A2 = {A2:.3e} [mm²]')
d2 = w.h/2 + s.h/2
print(f'd2 = {d2:.3e} [mm]')
Iz2 = Izc2 + A2*d2**2
print(f'Iz2 = {Iz2:.3e} [mm⁴]')

print('smaller rectangular area:')
Iz  = Iz1 + 2*Iz2
print(f'Iz (homog. section) = {Iz:.3e} [mm⁴]')

alma:
Iz (alma) :: (b,h) = (150.0,250.0) :: 1.953e+08 [mm⁴]
banzo:
Iz (banzo) :: (b,h) = (3150,10) ::  2.625e+05 [mm⁴]
A2 = 3.150e+04 [mm²]
d2 = 1.300e+02 [mm]
Iz2 = 5.326e+08 [mm⁴]
smaller rectangular area:
Iz (homog. section) = 1.261e+09 [mm⁴]


- Momento de área de primeira ordem (da área superior de aço tranformada em madeira):

\begin{equation*}
Q = A_1 \overline{y}_1
\end{equation*}

In [30]:
y_1 = w.h/2 + s.h/2
print(f'y = {y_1:.3f} [m]')
A_1 = b_s2w*s.h
Q = A_1*y_1
print(f'Q = {Q:.3e} [m³]')

y = 130.000 [m]
Q = 4.095e+06 [m³]


\begin{equation*}
\tau = \frac{V Q}{I_z b}
\quad\wedge\quad
q = \frac{V Q}{I_z} = \tau~b
\quad\wedge\quad
\tau = \frac{F}{b~d_p}
\quad\wedge\quad
q = \frac{F}{d_p}
\end{equation*}

\begin{equation*}
\quad\therefore\quad
\frac{F}{d_p} = \frac{V Q}{I_z}
\quad\Leftrightarrow\quad
V = \frac{F I_z}{Q d_p}
\end{equation*}

onde $d_p$ é o espaçamento entre pregos, pois a força de corte em cada prego, para um espaçamento de $d_p$ vem: $F = d_p ~\tau$.

In [33]:
v = sy.symbols('v')

q = d.Fcomax/d.palen
print(f'shear flow (shear per unit length) = {q:.1f} [MPa]')
V = solve(v - d.Fcomax*Iz/Q/d.palen,v)[0]
print(f'V = {V:.1f} [N]')

def tenscorte(V_,Q_,Iz_,b_):
    return V_*Q_/Iz_/b_

tau = tenscorte(V,Q,Iz,w.b)
print(f'tau = {tau:.1f} [MPa] < {s.tauadm} [MPa]')

# tau max na madeira ocorre na posição do eixo neutro:
Qmax_mad = Q + w.h/2*w.b*(w.h/2/2)
print(f'Qmax_mad = {Qmax_mad:.1f} [mm³]')
taumax_mad = tenscorte(V,Qmax_mad,Iz,w.b)
print(f'taumax.madeira = {taumax_mad:.3f} [MPa] < {w.tauadm} [MPa]')

flow shear = 75.0 [MPa]
V = 23086.8 [N]
tau = 0.5 [MPa] < 135 [MPa]
Qmax_mad = 5266875.0 [mm³]
taumax.madeira = 0.643 [MPa] < 7 [MPa]


---

Copyright (c) DEMI - FCT NOVA

Interactive computing by <a href="https://jupyter.org/" target="_blank"> <span
style="color:#333399"> Jupyter Notebook </span> </a> &nbsp;|&nbsp;Coded by <a href = "mailto: jmc.xavier@fct.unl.pt">José Xavier</a>

Licensed under  <a href="http://creativecommons.org/licenses/by-sa/4.0/"
target="_blank"> <span style="color:#333399;font-size: 20px"> CC BY-SA 4.0  </span></a>